In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from safetensors.torch import load_file as load_safetensors_file
except Exception:
    load_safetensors_file = None

BASE_DIR = Path('.')
MODELS_BASE = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'results' / 'model_diagnostics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    ('1_baseline', 'bert_9classes_final'),
    ('2_groupdro', 'bert_gdro_eta01_2ep'),
    ('3_scrubbing', 'bert_scrubbing'),
    ('4_oversampling', 'bert_oversample_only'),
    ('5_focal_loss', 'bert_focal_loss'),
    ('6_adversarial', 'bert_adversarial'),
    ('7_label_smoothing', 'bert_label_smoothing'),
    ('8_attribution_reg', 'bert_attr_reg'),
    ('9_combined_scrub_gdro', 'bert_debiased_combo'),
    ('10_combined_best', 'combined_scrubbing_groupdro'),
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Models base: {MODELS_BASE.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')


In [ ]:
def find_model_layout(model_dir):
    model_dir = Path(model_dir)
    if not model_dir.exists():
        return None, None
    if (model_dir / 'config.json').exists():
        return 'hf_root', model_dir
    if (model_dir / 'bert' / 'config.json').exists():
        return 'split_bert', model_dir
    candidates = sorted([sub for sub in model_dir.iterdir() if sub.is_dir() and (sub / 'config.json').exists()])
    if candidates:
        preferred = [sub for sub in candidates if sub.name == 'final']
        return 'nested_hf', preferred[0] if preferred else candidates[-1]
    return None, None


def find_weight_file(path):
    path = Path(path)
    for name in ['model.safetensors', 'pytorch_model.bin']:
        candidate = path / name
        if candidate.exists():
            return candidate
    return None


def read_state_keys(weight_file):
    if weight_file is None:
        return []
    weight_file = Path(weight_file)
    if weight_file.suffix == '.safetensors':
        if load_safetensors_file is None:
            raise RuntimeError('safetensors is not available')
        state = load_safetensors_file(str(weight_file), device='cpu')
        return sorted(state.keys())
    obj = torch.load(weight_file, map_location='cpu')
    if isinstance(obj, dict) and 'state_dict' in obj and isinstance(obj['state_dict'], dict):
        return sorted(obj['state_dict'].keys())
    if isinstance(obj, dict):
        return sorted(obj.keys())
    return []


def extract_classifier_keys(keys):
    direct = [k for k in keys if k.startswith('classifier.')]
    bare = [k for k in keys if k in {'weight', 'bias'}]
    return sorted(set(direct + bare))


def sidecar_candidates(model_dir):
    model_dir = Path(model_dir)
    return [model_dir / 'classifier.pt', model_dir / 'model.pt', model_dir / 'classifier.bin']


def try_load(layout, resolved_path):
    tokenizer = None
    model = None
    loading_info = {}
    missing_keys = []
    error = ''
    try:
        if layout == 'split_bert':
            bert_path = resolved_path / 'bert'
            tokenizer_path = resolved_path if (resolved_path / 'tokenizer.json').exists() else bert_path
            tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(bert_path), output_loading_info=True)
        else:
            tokenizer = AutoTokenizer.from_pretrained(str(resolved_path))
            model, loading_info = AutoModelForSequenceClassification.from_pretrained(str(resolved_path), output_loading_info=True)
        missing_keys = loading_info.get('missing_keys', []) or []
        model = model.to(device).eval()
        return True, tokenizer is not None, missing_keys, error
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
        return False, tokenizer is not None, missing_keys, error
    finally:
        del tokenizer
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_model_checks(model_name, folder_name):
    model_dir = MODELS_BASE / folder_name
    layout, resolved_path = find_model_layout(model_dir)
    result = {
        'model_name': model_name,
        'folder_name': folder_name,
        'model_dir_exists': model_dir.exists(),
        'layout': layout or 'missing',
        'resolved_path': str(resolved_path) if resolved_path else '',
        'config_exists': False,
        'main_weight_file': '',
        'main_classifier_keys': '',
        'sidecar_file': '',
        'sidecar_classifier_keys': '',
        'label_encoder_found': False,
        'tokenizer_loaded': False,
        'load_ok': False,
        'missing_classifier_on_load': False,
        'missing_keys': '',
        'error': '',
    }
    if resolved_path is None:
        return result

    config_path = resolved_path / 'config.json' if layout != 'split_bert' else resolved_path / 'bert' / 'config.json'
    result['config_exists'] = config_path.exists()

    main_weights_base = resolved_path if layout != 'split_bert' else resolved_path / 'bert'
    main_weight_file = find_weight_file(main_weights_base)
    result['main_weight_file'] = str(main_weight_file) if main_weight_file else ''
    if main_weight_file:
        try:
            result['main_classifier_keys'] = ', '.join(extract_classifier_keys(read_state_keys(main_weight_file)))
        except Exception as exc:
            result['main_classifier_keys'] = f'ERROR: {type(exc).__name__}: {exc}'

    sidecar_file = ''
    for candidate in sidecar_candidates(model_dir):
        if candidate.exists():
            sidecar_file = candidate
            break
    result['sidecar_file'] = str(sidecar_file) if sidecar_file else ''
    if sidecar_file:
        try:
            result['sidecar_classifier_keys'] = ', '.join(extract_classifier_keys(read_state_keys(sidecar_file)))
        except Exception as exc:
            result['sidecar_classifier_keys'] = f'ERROR: {type(exc).__name__}: {exc}'

    label_encoder_paths = [resolved_path / 'label_encoder.joblib', resolved_path.parent / 'label_encoder.joblib', model_dir / 'label_encoder.joblib']
    result['label_encoder_found'] = any(path.exists() for path in label_encoder_paths)

    load_ok, tokenizer_loaded, missing_keys, error = try_load(layout, resolved_path)
    result['tokenizer_loaded'] = tokenizer_loaded
    result['load_ok'] = load_ok
    result['missing_keys'] = ', '.join(missing_keys)
    result['missing_classifier_on_load'] = any(key.startswith('classifier.') for key in missing_keys)
    result['error'] = error
    return result


In [ ]:
result_1 = run_model_checks('1_baseline', 'bert_9classes_final')
pd.DataFrame([result_1]).to_csv(OUTPUT_DIR / '1_baseline.csv', index=False)
pd.DataFrame([result_1])


In [ ]:
result_2 = run_model_checks('2_groupdro', 'bert_gdro_eta01_2ep')
pd.DataFrame([result_2]).to_csv(OUTPUT_DIR / '2_groupdro.csv', index=False)
pd.DataFrame([result_2])


In [ ]:
result_3 = run_model_checks('3_scrubbing', 'bert_scrubbing')
pd.DataFrame([result_3]).to_csv(OUTPUT_DIR / '3_scrubbing.csv', index=False)
pd.DataFrame([result_3])


In [ ]:
result_4 = run_model_checks('4_oversampling', 'bert_oversample_only')
pd.DataFrame([result_4]).to_csv(OUTPUT_DIR / '4_oversampling.csv', index=False)
pd.DataFrame([result_4])


In [ ]:
result_5 = run_model_checks('5_focal_loss', 'bert_focal_loss')
pd.DataFrame([result_5]).to_csv(OUTPUT_DIR / '5_focal_loss.csv', index=False)
pd.DataFrame([result_5])


In [ ]:
result_6 = run_model_checks('6_adversarial', 'bert_adversarial')
pd.DataFrame([result_6]).to_csv(OUTPUT_DIR / '6_adversarial.csv', index=False)
pd.DataFrame([result_6])


In [ ]:
result_7 = run_model_checks('7_label_smoothing', 'bert_label_smoothing')
pd.DataFrame([result_7]).to_csv(OUTPUT_DIR / '7_label_smoothing.csv', index=False)
pd.DataFrame([result_7])


In [ ]:
result_8 = run_model_checks('8_attribution_reg', 'bert_attr_reg')
pd.DataFrame([result_8]).to_csv(OUTPUT_DIR / '8_attribution_reg.csv', index=False)
pd.DataFrame([result_8])


In [ ]:
result_9 = run_model_checks('9_combined_scrub_gdro', 'bert_debiased_combo')
pd.DataFrame([result_9]).to_csv(OUTPUT_DIR / '9_combined_scrub_gdro.csv', index=False)
pd.DataFrame([result_9])


In [ ]:
result_10 = run_model_checks('10_combined_best', 'combined_scrubbing_groupdro')
pd.DataFrame([result_10]).to_csv(OUTPUT_DIR / '10_combined_best.csv', index=False)
pd.DataFrame([result_10])


In [ ]:
csv_paths = [OUTPUT_DIR / f'{model_name}.csv' for model_name, _ in MODELS]
summary = pd.concat([pd.read_csv(path) for path in csv_paths if path.exists()], ignore_index=True)
summary.to_csv(OUTPUT_DIR / 'model_diagnostics_summary.csv', index=False)
summary
